# Sandbox - Hyper-Parameters optimisation

This notebook is used to prototype the Hyper-Parameters optimisation process, in all scenarios of use of the algorithm.

---

## Imports & Config

In [1]:
! pwd

/Users/simonlejoly/Documents/Work/mimosa/tests


In [2]:
! export XLA_PYTHON_CLIENT_MEM_FRACTION=.25

In [3]:
# Jax configuration
USE_JIT = True
USE_X64 = True
DEBUG_NANS = False
VERBOSE = False

In [4]:
# Standard library imports
import os
os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from typing import Tuple

In [5]:
# Third party
import jax
jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)
import jax.random as jr
import jax.numpy as jnp
import jax.lax as lax
import jax.scipy as jsp
from jax import vmap, jit, Array, grad

from equinox import filter_jit
import numpy as np

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.dpi'] = 300
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import numpy as np

from kernax import WhiteNoiseKernel, VarianceKernel, SEKernel, AffineMean, AbstractMean, AbstractKernel
from kernax.mask import create_mask

In [6]:
%matplotlib inline

In [7]:
# Local imports
from mimosa.linalg import cho_factor, cho_solve
from mimosa.generate_data import generate_data
from mimosa.hyperpost import hyperpost
from mimosa.sampling import sample_gp
from mimosa.nll import means_nlls, tasks_nlls, full_nll

2026-04-23 10:43:30,746 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [8]:
# Config
key = jr.PRNGKey(42)

T=120 ; K=3 ; F=1 ; N=75 ; I=1 ; O=2 ; gs=250 if I == 1 else 40

sth=True ; sch=True ; chit=True ; fh=False ; soh=True ; siit=True ; siif=True

mean = AffineMean(slope=0., intercept=0.)
mean_kernel = VarianceKernel(20.) * SEKernel(length_scale=10.)
task_kernel = VarianceKernel(.2) * SEKernel(length_scale=9.) + WhiteNoiseKernel(noise=.01)

mean_priors = {
	"slope": (-.2, .2),
	"intercept": (-2.5, 2.5)
}

mean_kernel_priors = {
	"variance": (5, 10.),
	"length_scale": (2.5, 10.)
}

task_kernel_priors = {
	"variance": (0.25, 1.),
	"length_scale": (2., 8.),
	"noise": (0.01, 0.1)
}

jax.devices()

[CpuDevice(id=0)]

In [9]:
inputs, outputs, maps, grid, m_p_means, m_p_covs, m_p, mix, t_m, m, m_k, t_k = generate_data(
	key, T, K, F, N,  I, O, gs,
	mean, mean_kernel, task_kernel,
	mean_priors, mean_kernel_priors, task_kernel_priors,
	sth, sch, chit, fh, soh, siit, siif)

In [10]:
mix_coeffs = jnp.eye(K)[mix]
mix_coeffs.shape

(120, 3)

In [11]:
p_m, p_c = hyperpost(inputs, outputs, maps, grid, mix_coeffs, m, m_k, t_k)
p_m.shape, p_c.shape

/opt/miniconda3/envs/mimosa/lib/python3.13/site-packages/jax/_src/scipy/linalg.py:167: FutureWarning: jax.scipy.linalg.cho_solve: batched 1D solves with b.ndim > 1 are deprecated, and in the future will be treated as a batched 2D solve. Use cho_solve(c_and_lower, b[..., None]).squeeze(-1) to avoid this warning.
  warnings.warn(


((3, 2, 250), (3, 1, 250, 250))

In [12]:
print(t_k)

VarianceKernel(variance=[0.78 ± 0.15]₃) * SEKernel(length_scale=[4.40 ± 2.13]₃) + WhiteNoiseKernel(noise=[0.04 ± 0.03]₃)


## Prediction

In [13]:
def predict_task_output(output_obs, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid):
	""" Predicts a single task output, within a single cluster"""
	padding_mask_1D = ~jnp.isnan(output_obs)[:, None]
	padding_mask_2D = padding_mask_1D & padding_mask_1D.T

	gamma_obs = jnp.where(padding_mask_2D, gamma_obs, jnp.eye(len(gamma_obs)))
	gamma_crossed = jnp.where(padding_mask_1D, gamma_crossed, 0.)

	gamma_pred_U = cho_factor(gamma_obs)
	z = lax.linalg.triangular_solve(gamma_pred_U, gamma_crossed.T).T
	y = lax.linalg.triangular_solve(gamma_pred_U, jnp.nan_to_num(output_obs) - post_mean_obs)

	pred_mean = post_mean_grid + (z.T @ y)
	pred_cov = gamma_grid - (z.T @ z)

	return pred_mean, pred_cov

In [14]:
def predict_task_in_cluster(output_obs, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid):
	if gamma_obs.shape[0] == 1:
		return (vmap(
			predict_task_output,
			in_axes=(0, 0, 0, None, None, None))
		        (output_obs.T, post_mean_obs, post_mean_grid, gamma_obs[0], gamma_crossed[0], gamma_grid[0]))
	else:
		return (vmap(
			predict_task_output,
			in_axes=(0, 0, 0, 0, 0, 0))
		        (output_obs.T, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid))

In [15]:
def predict_clusters(task_outputs, task_mappings,
                     post_mean_grid,
                     post_cov_grid,
                     task_cov_obs, task_cov_grid, task_cov_crossed):
	post_mean_obs = post_mean_grid[:, :, task_mappings]
	post_cov_obs = post_cov_grid[:, :, task_mappings, :][:, :, :, task_mappings]
	post_cov_crossed = post_cov_grid[:, :, task_mappings, :]

	gamma_obs = post_cov_obs + task_cov_obs
	gamma_grid = post_cov_grid + task_cov_grid
	gamma_crossed = post_cov_crossed + task_cov_crossed

	if gamma_obs.shape[0] == 1:
		return vmap(
			predict_task_in_cluster,
			in_axes=(None, 0, 0, None, None, None)
		)(task_outputs, post_mean_obs, post_mean_grid, gamma_obs[0], gamma_crossed[0], gamma_grid[0])
	return vmap(
		predict_task_in_cluster,
		in_axes=(None, 0, 0, 0, 0, 0)
	)(task_outputs, post_mean_obs, post_mean_grid, gamma_obs, gamma_crossed, gamma_grid)

In [16]:
def predict(outputs, mappings,
            post_mean_grid,
            post_cov_grid,
            tasks_cov_obs, tasks_cov_grid, tasks_cov_crossed):
	""" Predict every task for every cluster. """
	if tasks_cov_obs.shape[0] == 1:
		return vmap(
			predict_clusters,
			in_axes=(0, 0 if mappings.ndim == 2 else None,
			         None,
			         None,
			         None, None, None),
		)(outputs, mappings,
		  post_mean_grid,
		  post_cov_grid,
		  tasks_cov_obs[0], tasks_cov_grid[0], tasks_cov_crossed[0])

	return vmap(
		predict_clusters,
		in_axes=(0, 0 if mappings.ndim == 2 else None,
		         None,
		         None,
		         0, 0, 0),
	)(outputs, mappings,
	  post_mean_grid,
	  post_cov_grid,
	  tasks_cov_obs, tasks_cov_grid, tasks_cov_crossed)

In [17]:
t_k(inputs).shape, t_k(grid).shape

((1, 3, 1, 75, 75), (1, 3, 1, 250, 250))

In [18]:
t_k_noiseless = t_k.replace(noise=1e-12)
if not siit:
	extended_grid = jnp.broadcast_to(grid, (len(inputs),)+grid.shape)
else:
	extended_grid = grid

In [19]:
outputs.shape, maps.shape

((120, 75, 2), (75,))

In [20]:
pred_means, pred_covs = predict(outputs, maps, p_m, p_c, t_k(inputs), t_k_noiseless(extended_grid), t_k_noiseless(inputs, extended_grid))
pred_means.shape, pred_covs.shape

((120, 3, 2, 250), (120, 3, 2, 250, 250))

In [21]:
# Shape match : (Task, Cluster, Output, Grid, Grid)

In [22]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

grid_x = np.array(grid[:, 0])
m_prior = np.array(m(grid))  # (K, O, gs) — true hyperprior means
m_post  = np.array(m_p)      # (K, O, gs) — true hyperposterior means

def plot_task(t=0):
    row_titles = [f"Cluster {k}  (p={float(mix_coeffs[t, k]):.3f})" for k in range(K)]
    col_titles  = [f"Output {o}" for o in range(O)]

    fig = make_subplots(
        rows=K, cols=O,
        row_titles=row_titles,
        column_titles=col_titles,
    )

    for k in range(K):
        for o in range(O):
            show = (k == 0) and (o == 0)
            obs_x = np.array(inputs[t, :, 0])
            obs_y = np.array(outputs[t, :, o])
            pm    = np.array(pred_means[t, k, o])
            pc    = np.array(jnp.sqrt(jnp.abs(jnp.diag(pred_covs[t, k, o]))))

            fig.add_trace(go.Scatter(
                x=obs_x, y=obs_y,
                mode='markers',
                marker=dict(color='green', size=4, opacity=0.7),
                name='Observations',
                showlegend=show, legendgroup='obs',
            ), row=k+1, col=o+1)

            fig.add_trace(go.Scatter(
                x=grid_x, y=m_prior[0 if sch else k, 0 if soh else o] if sch else [k, o],
                mode='lines',
                line=dict(color='blue', dash='dash', width=1),
                opacity=0.3,
                name='Hyperprior mean',
                showlegend=show, legendgroup='hprior',
            ), row=k+1, col=o+1)

            fig.add_trace(go.Scatter(
                x=grid_x, y=m_post[k, o],
                mode='lines',
                line=dict(color='blue', width=1.5),
                opacity=0.5,
                name='Hyperposterior mean',
                showlegend=show, legendgroup='hpost',
            ), row=k+1, col=o+1)

            fig.add_trace(go.Scatter(
                x=np.concatenate([grid_x, grid_x[::-1]]),
                y=np.concatenate([pm + 1.98*pc, (pm - 1.98*pc)[::-1]]),
                fill='toself',
                fillcolor='rgba(0,180,0,0.15)',
                line=dict(color='rgba(0,0,0,0)'),
                name='95% CI',
                showlegend=show, legendgroup='ci',
            ), row=k+1, col=o+1)

            fig.add_trace(go.Scatter(
                x=grid_x, y=pm,
                mode='lines',
                line=dict(color='green', width=2),
                name='Prediction mean',
                showlegend=show, legendgroup='pred',
            ), row=k+1, col=o+1)

    fig.update_layout(
        title=f"Task {t}  —  true cluster: {int(mix[t])}",
        height=350 * K,
        width=max(500 * O + 150, 800),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    display(fig)

widgets.interact(plot_task, t=widgets.IntSlider(min=0, max=T-1, step=1, value=0, description='Task:'))

interactive(children=(IntSlider(value=0, description='Task:', max=119), Output()), _dom_classes=('widget-inter…

<function __main__.plot_task(t=0)>

---